In [ ]:
import os
os.environ['PYQTGRAPH_QT_LIB'] = 'PyQt5'
from fobench import Fiber
%gui qt

# Signal Processing Basics
This tutorial will focus on the signal processing capabilities of FoBench. We already looked at loading data and some basic visualisations, so lets go right ahead and load another one of the example files:

In [ ]:
das = Fiber('./example_data/terra15_hdf5/Terra15_velocity_1.hdf5', 'terra15')
das

By looking at `units`, we can see that the file was recorded in the native velocity format of the Terra15 interrogator. FoBench offers a range of instrumental correction functionality. For this interrogator we need to specify a gauge legth that is being applied to the data. Let's convert the data into strain-rate using a gauge length of 5m. 

In [ ]:
das.instr_correct(target="strain-rate", terra15_gl=5)

## Filtering
Filtering is probably one of the most common operations when processing FOS data. FoBench offers the known Obspy filtering functions such as `bandpass`, `highpass`, `bandstop`, `lowpass` or the more specialized filters `remez_fir` or `lowpass_fir`. A few additional filters are implemented: the adaptive frequency-wavenumber filter (`afk`, [Isken et al. 2022](https://academic.oup.com/gji/article/231/2/944/6609779)), a `median`-filter and a `fk_filter`. Checkout the documentation of the filter package to learn more about the filtering options.
For our example lets go with a good old highpass:

In [ ]:
das.filter(f_type="highpass", freq=2)
das.processing

By checking `Fiber.processing` we can see the full history of processing steps that have been performed on the `Fiber` instance. In your scripts this lets you check at any point how the data held in the instace has been modified. If later you want to develope your own processing methods, decorate them with the `@_update_processing` decorator to log their calls to the `Fiber.processing` dictionary.

In our case here we can see that the `filter` method automatically calls some pre-processing steps that are applied before filtering. It always worth checking the [documentation](https://doctus5.github.io/fobench/) and check on all the parameters for methods you are using. The filter method for example can be called with `pre_process=False` to turn off pre-processing completely or skipping some of the pre-processing steps by modifiying `steps`.

## Demean, detrend and common mode removal
While the `filter` method already performs demeaning and detrending we can also perform these steps individually with `Fiber.demean()` and `Fiber.detrend()` respectively. `demean` can remove both `"mean"` and `"median"` via the mode parameter and `detrend` can also remove higher order polynomials as trends, controlled through the `order` argument.
To remove common mode noise, FoBench offers the convenience method `remove_common_mode` which just wraps `Fiber.demean(mode=mode, dim="d")`.


In [ ]:
das.remove_common_mode()

## Normalizing
Your FOS data can be normalized in a few different ways. We can normalize with respect to the whole record (`"absolute max"`) or for each channel individually (`"trace max"`). FoBench also offers running absolute mean (`"running mean"`) and `"1 bit"` normalization. After that, lets focus only on the last seconds of the file.


In [ ]:
das.normalize(method="absolute max")
das.trim(t0=das.start_time + 40, tf = das.end_time - 2)
das.view()

## Other processing methods and functions
Data can be **integrated** and **differentiated**, for example to move between strain and strain rate.

In [ ]:
print(das.units)
das.integrate()
print(das.units)
das.differentiate()
print(das.units)
#das.taper()

Ambient noise processing often requires spectral whitening. Call `Fiber.whiten(freq_min, freq_max)` to whiten the spectrum of the record between `freq_min` and `freq_max` Hz. 

In [ ]:
das.spectrum([10, 20, 30])
das.whiten(freq_min=5, freq_max=25)
das.spectrum([10, 20, 30])

The [`signals` module](https://doctus5.github.io/fobench/api/fobench.core.tools.signals.html#module-fobench.core.tools.signals) has more functionality available that can be applied to any kind of signal. These include:

- Hilbert transform and signal envelope 
- Tapering
- Channel Spectrogram computation
- Continious Wavelet transform computation and plotting

## The `dim` parameter

Many of the methods and functions in FoBench have a `dim` argument. It controls along which dimension of a 2D array computation is performed. Most signal processing is performed on time series default but can in theory can also be applied along the distance dimension by changing the dim argument to from `t` to `d`.